# Layer 1 — Risk Tier Classifier
**AI Financial Advisor · Group 2 · MIS 02.303**

This notebook trains the XGBoost risk tier classifier, audits for age bias via SHAP, and saves `model.pkl` for use by `predict.py`.

**Run order:** Cell by cell, top to bottom. Do not skip cells.

**Output:** `layer1_ml/model.pkl` — download and commit to the `layer1-ml` branch when done.

---
## 1. Install dependencies

In [ ]:
!pip install -q xgboost shap scikit-learn pandas matplotlib joblib

---
## 2. Load data

Upload `financial_risk_profiles.csv` from the `data/` folder when Colab prompts.

In [ ]:
from google.colab import files
import io, pandas as pd

print('Upload financial_risk_profiles.csv')
uploaded = files.upload()
df = pd.read_csv(io.BytesIO(list(uploaded.values())[0]))
print(f'Loaded {len(df):,} rows')
df.head()

---
## 3. Explore — class balance and feature distributions

In [ ]:
import matplotlib.pyplot as plt

print('=== Class distribution ===')
print(df['risk_tier'].value_counts())
print()
print(df['risk_tier'].value_counts(normalize=True).mul(100).round(1).astype(str) + '%')

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

df['risk_tier'].value_counts().plot(kind='bar', ax=axes[0], color=['#e74c3c','#f39c12','#27ae60'])
axes[0].set_title('Class distribution (raw)')
axes[0].set_xlabel('')

df.boxplot(column='age', by='risk_tier', ax=axes[1])
axes[1].set_title('Age by risk tier')
plt.suptitle('')
plt.tight_layout()
plt.show()

print()
print('=== Age stats by tier ===')
print(df.groupby('risk_tier')['age'].describe().round(1))

---
## 4. Feature engineering

**`age`** is kept as a spec-required feature.

**`years_to_retirement`** is added as an engineered feature (`65 - age`). A 25-year-old with a 10-year horizon and a 65-year-old with a 10-year horizon have identical horizons but very different time-to-need contexts. Keeping both lets the model capture that distinction, and SHAP will show which one is doing the work for any given prediction.

In [ ]:
import numpy as np

df['years_to_retirement'] = (65 - df['age']).clip(lower=0)

# Ordinal encode investment_experience: Beginner=0, Intermediate=1, Advanced=2
exp_map = {'Beginner': 0, 'Intermediate': 1, 'Advanced': 2}
df['experience_encoded'] = df['investment_experience'].map(exp_map)

# age (spec-required) + years_to_retirement (engineered, 65 - age)
FEATURE_COLS = [
    'age',
    'years_to_retirement',
    'annual_income_usd',
    'savings_rate_pct',
    'debt_to_income_ratio',
    'investment_horizon_years',
    'experience_encoded',
]

TARGET_COL = 'risk_tier'
LABEL_MAP  = {'Low': 0, 'Medium': 1, 'High': 2}
LABEL_INV  = {v: k for k, v in LABEL_MAP.items()}

X = df[FEATURE_COLS].values
y = df[TARGET_COL].map(LABEL_MAP).values

print('Features:', FEATURE_COLS)
print('X shape:', X.shape)
print('Class counts:', {LABEL_INV[k]: int(v) for k, v in zip(*np.unique(y, return_counts=True))})

---
## 5. Class imbalance strategy

The dataset has a significant class imbalance: Low=243 (12%), High=477 (24%), Medium=1280 (64%).

We handle this with **class weights** rather than SMOTE. SMOTE generates synthetic samples by interpolating between existing ones — for a spec-defined dataset used in a graded project, modifying the training distribution by adding synthetic rows could misrepresent the model's behavior during evaluation. Class weights achieve the same goal (preventing the model from ignoring the minority `Low` tier) without altering the dataset itself.

Class weights are computed from `y_train` after the split in Cell 6 so they reflect the actual training distribution, not the full dataset.

---
## 6. Train / test split + class weights

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.utils.class_weight import compute_class_weight
from collections import Counter

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y,
)

print(f'Train: {len(X_train):,} rows')
print(f'Test:  {len(X_test):,} rows')
print('Train class counts:', Counter({LABEL_INV[k]: v for k, v in Counter(y_train).items()}))
print('Test  class counts:', Counter({LABEL_INV[k]: v for k, v in Counter(y_test).items()}))
print()

# Compute weights from y_train so they reflect the actual training distribution
classes = np.unique(y_train)
weights = compute_class_weight('balanced', classes=classes, y=y_train)
weight_dict = dict(zip(classes, weights))
sample_weights = np.array([weight_dict[label] for label in y_train])

print('Class weights (higher = model penalizes misclassification more):')
for cls, w in weight_dict.items():
    print(f'  {LABEL_INV[cls]:<8} (label={cls}): {w:.4f}')

---
## 7. Train XGBoost classifier

In [ ]:
from xgboost import XGBClassifier

model = XGBClassifier(
    n_estimators=300,
    max_depth=5,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    eval_metric='mlogloss',
    random_state=42,
    n_jobs=-1,
)

model.fit(
    X_train, y_train,
    sample_weight=sample_weights,
    eval_set=[(X_test, y_test)],
    verbose=50,
)

print('Training complete.')

---
## 8. Evaluate — accuracy, per-class F1, confusion matrix

Pay close attention to `Low` precision and recall — this is the tier most likely to be underserved.

In [ ]:
from sklearn.metrics import classification_report, confusion_matrix, ConfusionMatrixDisplay

y_pred = model.predict(X_test)
target_names = ['Low', 'Medium', 'High']

print('=== Classification Report ===')
print(classification_report(y_test, y_pred, target_names=target_names))

fig, ax = plt.subplots(figsize=(6, 5))
ConfusionMatrixDisplay(
    confusion_matrix(y_test, y_pred),
    display_labels=target_names
).plot(ax=ax, colorbar=False, cmap='Blues')
ax.set_title('Confusion Matrix — test set')
plt.tight_layout()
plt.show()

---
## 9. SHAP — global feature importance

Shows which features drive predictions across all users. With both `age` and `years_to_retirement` in the model, SHAP will separate their individual contributions — if one is doing redundant work, its mean |SHAP| will be near zero.

In [ ]:
import shap

explainer = shap.TreeExplainer(model)
shap_values = explainer.shap_values(X_test)

# Note: newer SHAP versions return (n_samples, n_features, n_classes) array instead of a list.
# summary_plot handles both — run it and verify the visualization looks correct.
shap.summary_plot(
    shap_values,
    X_test,
    feature_names=FEATURE_COLS,
    plot_type='bar',
    class_names=['Low', 'Medium', 'High'],
    show=True,
)

print()
print('=== Mean |SHAP| per feature (averaged across all classes) ===')
if isinstance(shap_values, list):
    all_shap = np.array(shap_values)  # (n_classes, n_samples, n_features)
    mean_shap = np.abs(all_shap).mean(axis=(0, 1))
else:
    # newer SHAP: (n_samples, n_features, n_classes)
    mean_shap = np.abs(shap_values).mean(axis=(0, 2))

ranking = sorted(zip(FEATURE_COLS, mean_shap), key=lambda x: x[1], reverse=True)
for rank, (feat, val) in enumerate(ranking, 1):
    print(f'  {rank}. {feat:<30} {val:.4f}')

---
## 10. SHAP — individual explanation (waterfall plot)

This is what each user will see in the app — which features pushed their prediction toward or away from their predicted tier.

In [ ]:
# Change SAMPLE_IDX to explore different users
SAMPLE_IDX = 0

sample_x = X_test[SAMPLE_IDX:SAMPLE_IDX+1]
pred_class = int(model.predict(sample_x)[0])
pred_proba = model.predict_proba(sample_x)[0]

print(f'Predicted tier: {LABEL_INV[pred_class]}')
print(f'Confidence: Low={pred_proba[0]:.2%}  Medium={pred_proba[1]:.2%}  High={pred_proba[2]:.2%}')
print()

explainer_v2 = shap.TreeExplainer(model)
shap_exp = explainer_v2(sample_x)

# Modern SHAP API: shap_exp is an Explanation object with shape (1, n_features, n_classes)
# Falls back to explicit constructor if version doesn't support this indexing
try:
    shap.plots.waterfall(shap_exp[0, :, pred_class], max_display=7, show=True)
except Exception:
    shap.plots.waterfall(
        shap.Explanation(
            values=shap_exp.values[0, :, pred_class],
            base_values=shap_exp.base_values[0, pred_class],
            data=sample_x[0],
            feature_names=FEATURE_COLS,
        ),
        max_display=7,
        show=True,
    )

print()
print('Top factors for this prediction:')
sv = shap_exp.values[0, :, pred_class]
top3 = sorted(zip(FEATURE_COLS, sv), key=lambda x: abs(x[1]), reverse=True)[:3]
for feat, impact in top3:
    direction = 'pushes toward this tier' if impact > 0 else 'pushes away from this tier'
    print(f'  {feat}: {impact:+.4f} ({direction})')

---
## 11. Save model.pkl

The bundle saved here matches exactly what `predict.py` loads at runtime.

In [ ]:
import joblib
from pathlib import Path

model_bundle = {
    'model':        model,
    'feature_cols': FEATURE_COLS,
    'label_map':    LABEL_MAP,
    'label_inv':    LABEL_INV,
    'exp_map':      exp_map,
    'notes':        'XGBoost · class-weight balanced · age + years_to_retirement (engineered)',
}

MODEL_PATH = Path('model.pkl')
joblib.dump(model_bundle, MODEL_PATH)
print(f'Saved {MODEL_PATH} ({MODEL_PATH.stat().st_size / 1024:.1f} KB)')

files.download(str(MODEL_PATH))
print('Download started — place model.pkl in layer1_ml/ in your local repo.')

---
## 12. Sanity-check load (mirrors what predict.py does)

Confirms the bundle loads correctly and returns the right shape for a real app call.

In [ ]:
bundle = joblib.load(MODEL_PATH)
loaded_model   = bundle['model']
loaded_feats   = bundle['feature_cols']
loaded_exp_map = bundle['exp_map']
loaded_inv     = bundle['label_inv']

# Simulate a real user profile coming in from the app
test_profile = {
    'age': 34,
    'annual_income_usd': 72000,
    'savings_rate_pct': 14.5,
    'debt_to_income_ratio': 0.28,
    'investment_horizon_years': 25,
    'investment_experience': 'Intermediate',
}

# 7-feature vector — must match FEATURE_COLS order exactly
row = [
    test_profile['age'],
    max(0, 65 - test_profile['age']),                        # years_to_retirement
    test_profile['annual_income_usd'],
    test_profile['savings_rate_pct'],
    test_profile['debt_to_income_ratio'],
    test_profile['investment_horizon_years'],
    loaded_exp_map[test_profile['investment_experience']],
]

X_single = np.array([row])
pred     = int(loaded_model.predict(X_single)[0])
proba    = loaded_model.predict_proba(X_single)[0]

print('Profile:', test_profile)
print()
print(f'Predicted tier : {loaded_inv[pred]}')
print(f'Confidence     : Low={proba[0]:.2%}  Medium={proba[1]:.2%}  High={proba[2]:.2%}')
print()
print('Feature vector :', list(zip(loaded_feats, row)))
print('Bundle keys    :', list(bundle.keys()))
print()
print('Sanity check PASSED')

---
## 13. Record metrics for model_card.md

Copy this output into `layer4_respai/model_card.md` Section 3.

In [ ]:
from sklearn.metrics import accuracy_score

acc = accuracy_score(y_test, y_pred)

print('======= COPY INTO model_card.md Section 3 =======')
print(f'- Overall accuracy: {acc:.1%}')
print(f'- Training set size: {len(X_train):,}')
print(f'- Test set size:     {len(X_test):,}')
print(f'- Model:             XGBoost (n_estimators=300, max_depth=5)')
print(f'- Class imbalance:   handled via compute_class_weight balanced (from y_train)')
print(f'- Features (7):      age, years_to_retirement, annual_income_usd,')
print(f'                     savings_rate_pct, debt_to_income_ratio,')
print(f'                     investment_horizon_years, experience_encoded')
print()
print('Per-class results:')
print(classification_report(y_test, y_pred, target_names=['Low', 'Medium', 'High']))
print('=================================================')